# 📓 Notebook 15 — Evaluation---**Phase 6 · Production Engineering**> If you can't measure it, you can't improve it. RAG and agent evaluation is what separates prototypes from production.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| RAG metrics | Faithfulness, relevance, retrieval accuracy || Hallucination detection | Catch when LLMs make things up || LLM-as-Judge | Use LLMs to evaluate LLM outputs || Test datasets | Build golden datasets for evaluation |### 🏗️ Build: RAG Evaluation Harness

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. RAG Evaluation Framework### The Three Pillars of RAG Quality```         ┌────────────────┐         │    Question     │         └───────┬────────┘                 ↓         ┌───────────────┐        ┌─────────────────────────┐         │   Retriever    │───────→│  RETRIEVAL METRICS       │         └───────┬───────┘        │  • Precision@K           │                 ↓                │  • Recall@K              │         ┌───────────────┐        │  • NDCG                  │         │   Context      │       └─────────────────────────┘         └───────┬───────┘                 ↓         ┌───────────────┐        ┌─────────────────────────┐         │   Generator    │───────→│  GENERATION METRICS      │         └───────┬───────┘        │  • Faithfulness           │                 ↓                │  • Answer Relevance       │         ┌───────────────┐        │  • Hallucination Score    │         │    Answer      │       └─────────────────────────┘         └───────────────┘```> **Interview Critical:** Faithfulness measures if the answer is supported by the retrieved context. Relevance measures if the answer actually addresses the question. Both can be high while the other is low.

In [ ]:
class RAGEvaluator:    """Evaluate RAG pipeline outputs across multiple dimensions."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        def evaluate_faithfulness(self, answer, context):        """Is the answer faithfully derived from the context? (No hallucination)"""        response = litellm.completion(            model=self.model,            messages=[{                "role": "system",                "content": "You evaluate if an answer is faithfully derived from the given context."            }, {                "role": "user",                "content": f"""Rate how faithfully the answer is supported by the context.Context: {context}Answer: {answer}Score from 0.0 to 1.0 where:- 1.0 = Every claim in the answer is directly supported by the context- 0.5 = Some claims are supported, some are not- 0.0 = The answer is completely unrelated to the contextRespond in JSON: {{"score": <float>, "reasoning": "<brief explanation>", "unsupported_claims": [<list of claims not in context>]}}"""            }],            response_format={"type": "json_object"},            temperature=0,        )        return json.loads(response.choices[0].message.content)        def evaluate_relevance(self, answer, question):        """Does the answer actually address the question?"""        response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"""Rate how well the answer addresses the question.Question: {question}Answer: {answer}Score from 0.0 to 1.0:- 1.0 = Fully and directly answers the question- 0.5 = Partially addresses the question- 0.0 = Doesn't answer the question at allRespond in JSON: {{"score": <float>, "reasoning": "<brief explanation>"}}"""            }],            response_format={"type": "json_object"},            temperature=0,        )        return json.loads(response.choices[0].message.content)        def evaluate_retrieval(self, question, retrieved_chunks, ground_truth_chunks):        """Measure retrieval quality (precision and recall)."""        # Use LLM to judge relevance of each chunk        relevant = 0        for chunk in retrieved_chunks:            response = litellm.completion(                model=self.model,                messages=[{                    "role": "user",                    "content": f"Is this text relevant to answering the question? Respond 'yes' or 'no'.\n\nQuestion: {question}\nText: {chunk}"                }],                temperature=0, max_tokens=10,            )            if "yes" in response.choices[0].message.content.lower():                relevant += 1                precision = relevant / len(retrieved_chunks) if retrieved_chunks else 0        recall = relevant / len(ground_truth_chunks) if ground_truth_chunks else 0        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0                return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}        def detect_hallucination(self, answer, context):        """Specifically detect hallucinated facts."""        response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"""Identify any facts in the answer that are NOT supported by the context.Context: {context}Answer: {answer}Return JSON:{{    "has_hallucination": <boolean>,    "hallucinated_facts": [<list of unsupported claims>],    "severity": "none|low|medium|high"}}"""            }],            response_format={"type": "json_object"},            temperature=0,        )        return json.loads(response.choices[0].message.content)evaluator = RAGEvaluator()print("✅ RAG Evaluator ready")

In [ ]:
# Test the evaluator# Case 1: Faithful answercontext_1 = "Python was created by Guido van Rossum in 1991. It emphasizes code readability."answer_1 = "Python was created by Guido van Rossum in 1991 and is known for its readable code."question_1 = "Who created Python?"# Case 2: Hallucinated answercontext_2 = "Python was created by Guido van Rossum in 1991."answer_2 = "Python was created by Guido van Rossum in 1991. It was initially developed at Bell Labs as part of the Unix project."print("📊 RAG Evaluation Demo")print("=" * 60)# Faithfulnessprint("\n--- Faithfulness ---")f1 = evaluator.evaluate_faithfulness(answer_1, context_1)print(f"  ✅ Good answer: {f1['score']:.2f} — {f1['reasoning']}")f2 = evaluator.evaluate_faithfulness(answer_2, context_2)print(f"  ⚠️ Hallucinated: {f2['score']:.2f} — {f2['reasoning']}")# Relevanceprint("\n--- Relevance ---")r1 = evaluator.evaluate_relevance(answer_1, question_1)print(f"  Score: {r1['score']:.2f} — {r1['reasoning']}")# Hallucination detectionprint("\n--- Hallucination Detection ---")h1 = evaluator.detect_hallucination(answer_2, context_2)print(f"  Has hallucination: {h1['has_hallucination']}")print(f"  Severity: {h1['severity']}")print(f"  Hallucinated facts: {h1.get('hallucinated_facts', [])}")

In [ ]:
# Build a test dataset and run batch evaluationtest_cases = [    {        "question": "What is the capital of France?",        "context": "France is a country in Western Europe. Its capital is Paris, known for the Eiffel Tower.",        "answer": "The capital of France is Paris.",        "label": "good"    },    {        "question": "When was Python created?",        "context": "Python was created by Guido van Rossum and first released in 1991.",        "answer": "Python was created in 1991 by Guido van Rossum at Google.",        "label": "hallucinated"    },    {        "question": "What is machine learning?",        "context": "Machine learning is a subset of AI that enables computers to learn from data.",        "answer": "The weather forecast for today shows rain.",        "label": "irrelevant"    },]print("📊 Batch Evaluation Results")print("=" * 60)for i, tc in enumerate(test_cases, 1):    faith = evaluator.evaluate_faithfulness(tc["answer"], tc["context"])    relevance = evaluator.evaluate_relevance(tc["answer"], tc["question"])    hallucination = evaluator.detect_hallucination(tc["answer"], tc["context"])        print(f"\n  Test {i} [{tc['label']}]:")    print(f"    Q: {tc['question']}")    print(f"    Faithfulness: {faith['score']:.2f}")    print(f"    Relevance:    {relevance['score']:.2f}")    print(f"    Hallucinated: {hallucination['has_hallucination']} ({hallucination['severity']})")

---## 📝 Interview Quiz — Evaluation### Q1: What is faithfulness in RAG evaluation? How is it different from relevance?<details><summary>💡 Answer</summary>**Faithfulness**: Is the answer supported by the retrieved context? Does it only state things found in the context?- High faithfulness = no hallucination- Measured by checking each claim against the context**Relevance**: Does the answer address the user's question?- Can be relevant but unfaithful (answers correctly but makes up facts)- Can be faithful but irrelevant (quotes context accurately but doesn't answer the question)**Both are needed:** You want answers that are faithful to context AND relevant to the question.</details>### Q2: What is "LLM-as-Judge"? What are its limitations?<details><summary>💡 Answer</summary>**LLM-as-Judge**: Using an LLM to evaluate another LLM's output. The evaluator LLM rates quality, detects hallucinations, checks relevance.**Limitations:**1. **Self-bias**: LLMs prefer LLM-generated text over human text2. **Position bias**: Prefers the first option in comparisons3. **Verbosity bias**: Rates longer answers higher regardless of quality4. **Inconsistency**: Same evaluation prompt can give different scores5. **Can't catch subtle errors**: May miss factual inaccuracies in its own knowledge blind spots**Mitigation**: Use multiple judges, calibrate with human annotations, use structured rubrics.</details>### Q3: How would you build a golden test dataset for RAG?<details><summary>💡 Answer</summary>1. **Source**: Take real documents from your corpus2. **Questions**: Generate diverse questions (simple/complex, factual/analytical)3. **Ground truth answers**: Human-written or verified answers4. **Relevant passages**: Tag which passages contain the answer5. **Edge cases**: Include unanswerable questions, ambiguous queries6. **Size**: 100-500 examples for reliable metrics7. **Diversity**: Cover all topic areas and question typesAutomate question generation with LLM, but ALWAYS have humans verify the ground truth.</details>### Q4: What metrics would you track in a production RAG dashboard?<details><summary>💡 Answer</summary>1. **Retrieval**: Precision@K, NDCG, average similarity score2. **Generation**: Faithfulness, relevance, hallucination rate3. **Latency**: P50, P95, P99 for retrieval and generation4. **Usage**: Queries/day, unique users, topic distribution5. **Failures**: Timeout rate, empty retrieval rate, error rate6. **Cost**: Tokens consumed, embedding calls, LLM calls7. **User satisfaction**: Thumbs up/down, retry rate</details>

---## ✅ Notebook 15 Summary| Concept | Key Takeaway ||---------|-------------|| Faithfulness | Is the answer supported by the context? (No hallucination) || Relevance | Does the answer address the question? || LLM-as-Judge | Use LLMs to evaluate LLMs, but beware biases || Hallucination detection | Check each claim against source context || Test datasets | Human-verified ground truth for reliable metrics |### ➡️ Next: [Notebook 16 — Guardrails & Security](./16_guardrails.ipynb)